# Projeto Final - Machine Learning

## Previsão do Valor de Financiamento Habitacional do Programa Minha Casa Minha Vida (MCMV)

### Contexto

O Programa Minha Casa Minha Vida (MCMV) é uma das principais iniciativas de política habitacional do Brasil, voltada à ampliação do acesso à moradia por meio de mecanismos de financiamento e subsídios habitacionais.

A base utilizada neste projeto contém aproximadamente 7,8 milhões de registros de operações de financiamento habitacional e 21 variáveis. Entre as informações disponíveis estão localização geográfica, renda familiar, valor de compra do imóvel, subsídios, taxa de juros e características da operação.

### Problema

Dadas as características disponíveis das operações de financiamento habitacional, é possível estimar o valor do financiamento concedido no Programa Minha Casa Minha Vida?

Além da capacidade preditiva, o projeto busca investigar quais características disponíveis na base apresentam maior contribuição para a estimativa do valor financiado.

### Objetivo

Desenvolver e avaliar modelos de Machine Learning capazes de estimar o valor do financiamento habitacional concedido no âmbito do Programa Minha Casa Minha Vida, utilizando as características disponíveis das operações.

### Variável-alvo (Target)

A variável-alvo será:

`vlr_financiamento`

Essa variável representa o valor do financiamento habitacional e constitui a variável numérica que os modelos tentarão estimar.

### Tipo de problema

- Aprendizado supervisionado
- Regressão

O problema é caracterizado como regressão porque a variável-alvo `vlr_financiamento` é uma variável numérica contínua.

### Variáveis candidatas

A base contém informações potencialmente explicativas relacionadas a:

- localização geográfica;
- renda familiar;
- valor de compra do imóvel;
- subsídios concedidos;
- taxa de juros;
- programa habitacional;
- tipo do imóvel;
- condição de cotista;
- sistema de amortização;
- compatibilidade com faixa de renda;
- características temporais da operação.

As variáveis efetivamente utilizadas nos modelos serão definidas após a análise exploratória dos dados e a avaliação de possível vazamento de informações (data leakage).

Uma variável somente deverá ser utilizada como preditora caso sua utilização seja metodologicamente adequada ao momento em que a previsão seria realizada.

### Hipótese inicial

Parte-se da hipótese de que características econômicas, geográficas e contratuais das operações apresentam capacidade preditiva sobre o valor do financiamento.

Será investigada especialmente a relação entre `vlr_financiamento` e variáveis como renda familiar, valor de compra do imóvel, subsídios, localização e condições da operação.

Essa hipótese será avaliada empiricamente durante a análise exploratória e a modelagem, sem pressupor previamente quais variáveis deverão compor o modelo final.

### Estratégia de modelagem

Inicialmente será utilizado um modelo simples como baseline, estabelecendo uma referência mínima de desempenho.

Posteriormente, diferentes algoritmos de regressão serão treinados e comparados.

O projeto buscará avaliar não apenas o desempenho preditivo, mas também aspectos relacionados a:

- generalização do modelo;
- overfitting;
- vazamento de informações;
- pré-processamento das variáveis;
- interpretação dos resultados;
- limitações do modelo.

### Métricas de avaliação

Os modelos serão avaliados principalmente pelas seguintes métricas:

- **MAE (Mean Absolute Error):** erro absoluto médio das previsões;
- **RMSE (Root Mean Squared Error):** medida de erro que penaliza mais fortemente erros elevados;
- **R² (Coeficiente de Determinação):** proporção da variabilidade da variável-alvo explicada pelo modelo.

O MAE terá especial relevância por permitir interpretar o erro médio do modelo diretamente em reais (R$), mesma unidade da variável `vlr_financiamento`.

### Resultado esperado

Espera-se construir um fluxo completo de Machine Learning, abrangendo análise exploratória, pré-processamento, treinamento, validação e comparação de modelos de regressão.

Ao final, pretende-se identificar o desempenho dos modelos, suas principais limitações e as características das operações que mais contribuem para a previsão do valor financiado.

## 2. Fonte, Obtenção e Validação dos Dados

### Fonte dos dados

A base utilizada contém dados do Programa Minha Casa Minha Vida (MCMV), disponibilizados pelo Ministério das Cidades.

O conjunto de dados é distribuído em arquivo compactado no formato ZIP e contém registros de operações de financiamento habitacional.

### Aquisição dos dados

Para tornar o projeto reproduzível, a obtenção da base foi automatizada no próprio notebook.

O processo consiste em:

1. verificar se a base já está disponível localmente;
2. realizar o download do arquivo compactado quando necessário;
3. validar o arquivo obtido;
4. descompactar o CSV automaticamente;
5. manter os arquivos de dados fora do versionamento Git.

### Validação inicial

Antes da análise exploratória, serão verificados:

- existência e tamanho do arquivo;
- estrutura do conjunto de dados;
- quantidade de variáveis;
- nomes das colunas;
- tipos inicialmente identificados;
- capacidade de leitura adequada do CSV.

Nesta etapa não serão realizadas transformações nos dados.

In [9]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile

# URL oficial da base
URL = "https://www.cidades.gov.br/images/stories/ArquivosSNH/ArquivosZIP/mcmv_financ_analitico_20260724.zip"

# Como o notebook está em /notebooks, a raiz é um nível acima
PROJETO = Path.cwd().parent
PASTA_DADOS = PROJETO / "data"

ARQUIVO_ZIP = PASTA_DADOS / "mcmv_financ_analitico_20260724.zip"
ARQUIVO_CSV = PASTA_DADOS / "mcmv_financ_analitico_20260724.csv"

# Cria a pasta data, caso ainda não exista
PASTA_DADOS.mkdir(parents=True, exist_ok=True)

# Baixa somente se o CSV ainda não estiver disponível
if not ARQUIVO_CSV.exists():

    if not ARQUIVO_ZIP.exists():
        print("Baixando a base MCMV...")
        urlretrieve(URL, ARQUIVO_ZIP)
        print("Download concluído.")

    print("Descompactando a base...")

    with zipfile.ZipFile(ARQUIVO_ZIP, "r") as zip_ref:
        zip_ref.extractall(PASTA_DADOS)

    print("Descompactação concluída.")

else:
    print("Base já disponível localmente.")

print(f"Arquivo: {ARQUIVO_CSV}")

Base já disponível localmente.
Arquivo: c:\Users\c082909\Documents\Projetos\mcmv-machine-learning\data\mcmv_financ_analitico_20260724.csv


In [11]:
import pandas as pd

# Verificação do arquivo
if not ARQUIVO_CSV.exists():
    raise FileNotFoundError(f"Base não encontrada: {ARQUIVO_CSV}")

tamanho_gb = ARQUIVO_CSV.stat().st_size / (1024 ** 3)

print("VALIDAÇÃO DO ARQUIVO")
print("-" * 50)
print(f"Arquivo: {ARQUIVO_CSV.name}")
print(f"Tamanho: {tamanho_gb:.2f} GB")

# Leitura de uma pequena amostra
amostra = pd.read_csv(
    ARQUIVO_CSV,
    sep=";",
    encoding="utf-8",
    nrows=1000,
    low_memory=False
)

print(f"Registros da amostra: {len(amostra):,}")
print(f"Quantidade de variáveis: {amostra.shape[1]}")

print("\nTIPOS IDENTIFICADOS NA AMOSTRA")
print("-" * 50)

display(
    amostra.dtypes
    .rename("tipo")
    .to_frame()
)

VALIDAÇÃO DO ARQUIVO
--------------------------------------------------
Arquivo: mcmv_financ_analitico_20260724.csv
Tamanho: 1.21 GB
Registros da amostra: 1,000
Quantidade de variáveis: 21

TIPOS IDENTIFICADOS NA AMOSTRA
--------------------------------------------------


,tipo
data_referencia,str
cod_ibge,int64
txt_municipio,str
mcmv_fgts_txt_uf,str
txt_regiao,str
data_assinatura_financiamento,str
qtd_uh_financiadas,int64
vlr_financiamento,str
vlr_subsidio_desconto_fgts,float64
vlr_subsidio_desconto_ogu,int64


In [12]:
colunas_inspecao = [
    "data_referencia",
    "data_assinatura_financiamento",
    "vlr_financiamento",
    "vlr_subsidio_desconto_fgts",
    "vlr_subsidio_desconto_ogu",
    "vlr_subsidio_equilíbrio_fgts",
    "vlr_subsidio_equilíbrio_ogu",
    "vlr_compra",
    "vlr_renda_familiar",
    "num_taxa_juros",
    "txt_compatibilidade_faixa_renda"
]

for coluna in colunas_inspecao:
    print(f"\n{coluna}")
    print("-" * 50)

    print(
        amostra[coluna]
        .dropna()
        .head(10)
        .tolist()
    )


data_referencia
--------------------------------------------------
['24/07/2026', '24/07/2026', '24/07/2026', '24/07/2026', '24/07/2026', '24/07/2026', '24/07/2026', '24/07/2026', '24/07/2026', '24/07/2026']

data_assinatura_financiamento
--------------------------------------------------
['2026-02-20 00:00:00.000', '2026-03-06 00:00:00.000', '2026-06-24 00:00:00.000', '2026-05-13 00:00:00.000', '2026-05-12 00:00:00.000', '2026-03-06 00:00:00.000', '2026-03-06 00:00:00.000', '2026-03-06 00:00:00.000', '2026-03-06 00:00:00.000', '2026-05-11 00:00:00.000']

vlr_financiamento
--------------------------------------------------
['144.415,57', '257.613,8', '252.367,3', '255.789,84', '279.715,76', '219.421', '194.964', '127.320,33', '108.901,59', '255.789,84']

vlr_subsidio_desconto_fgts
--------------------------------------------------
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

vlr_subsidio_desconto_ogu
--------------------------------------------------
[0, 0, 0, 0, 0, 0, 0, 0, 0,

In [13]:
amostra_validada = pd.read_csv(
    ARQUIVO_CSV,
    sep=";",
    encoding="utf-8",
    nrows=1000,
    decimal=",",
    thousands=".",
    low_memory=False
)

colunas_numericas = [
    "vlr_financiamento",
    "vlr_subsidio_desconto_fgts",
    "vlr_subsidio_desconto_ogu",
    "vlr_subsidio_equilíbrio_fgts",
    "vlr_subsidio_equilíbrio_ogu",
    "vlr_compra",
    "vlr_renda_familiar",
    "num_taxa_juros"
]

print("VALIDAÇÃO DOS CAMPOS NUMÉRICOS")
print("-" * 50)

for coluna in colunas_numericas:
    print(
        f"{coluna:<35} "
        f"{str(amostra_validada[coluna].dtype):<10} "
        f"Exemplo: {amostra_validada[coluna].iloc[0]}"
    )

VALIDAÇÃO DOS CAMPOS NUMÉRICOS
--------------------------------------------------
vlr_financiamento                   float64    Exemplo: 144415.57
vlr_subsidio_desconto_fgts          int64      Exemplo: 0
vlr_subsidio_desconto_ogu           int64      Exemplo: 0
vlr_subsidio_equilíbrio_fgts        float64    Exemplo: 0.0
vlr_subsidio_equilíbrio_ogu         int64      Exemplo: 0
vlr_compra                          float64    Exemplo: 213540.02
vlr_renda_familiar                  float64    Exemplo: 6070.02
num_taxa_juros                      float64    Exemplo: 8.16


## 3. Análise Exploratória e Qualidade dos Dados

Nesta etapa será realizada a análise exploratória dos dados (EDA), com o objetivo de compreender a estrutura, qualidade e distribuição das informações disponíveis antes da construção dos modelos de Machine Learning.

Serão avaliados:

- dimensões do conjunto de dados;
- tipos das variáveis;
- valores ausentes;
- registros duplicados;
- cardinalidade das variáveis categóricas;
- estatísticas descritivas;
- consistência dos valores;
- distribuição da variável-alvo;
- possíveis valores extremos (outliers);
- relações entre as variáveis.

As decisões de tratamento e pré-processamento serão tomadas somente após essas análises.

In [14]:
import pandas as pd

df = pd.read_csv(
    ARQUIVO_CSV,
    sep=";",
    encoding="utf-8",
    decimal=",",
    thousands=".",
    low_memory=False
)

print("CARREGAMENTO DA BASE COMPLETA")
print("-" * 50)

print(f"Registros: {df.shape[0]:,}".replace(",", "."))
print(f"Variáveis: {df.shape[1]}")

memoria_gb = df.memory_usage(deep=True).sum() / (1024 ** 3)

print(f"Memória utilizada pelo DataFrame: {memoria_gb:.2f} GB")

CARREGAMENTO DA BASE COMPLETA
--------------------------------------------------
Registros: 7.849.882
Variáveis: 21
Memória utilizada pelo DataFrame: 4.89 GB


In [15]:
# Conversão da data de contratação - Análise Temporal
df["data_assinatura_financiamento"] = pd.to_datetime(
    df["data_assinatura_financiamento"],
    errors="coerce"
)

print("ANÁLISE TEMPORAL DAS CONTRATAÇÕES")
print("-" * 50)

print(
    "Primeira contratação:",
    df["data_assinatura_financiamento"].min()
)

print(
    "Última contratação:",
    df["data_assinatura_financiamento"].max()
)

print(
    "Datas inválidas/nulas:",
    df["data_assinatura_financiamento"].isna().sum()
)

antes_2020 = (
    df["data_assinatura_financiamento"] < "2020-01-01"
).sum()

desde_2020 = (
    df["data_assinatura_financiamento"] >= "2020-01-01"
).sum()

total = len(df)

print("\nDISTRIBUIÇÃO")
print("-" * 50)

print(
    f"Antes de 01/01/2020: "
    f"{antes_2020:,} "
    f"({antes_2020 / total * 100:.2f}%)"
)

print(
    f"De 01/01/2020 em diante: "
    f"{desde_2020:,} "
    f"({desde_2020 / total * 100:.2f}%)"
)

ANÁLISE TEMPORAL DAS CONTRATAÇÕES
--------------------------------------------------
Primeira contratação: 2009-01-02 00:00:00
Última contratação: 2026-07-23 00:00:00
Datas inválidas/nulas: 0

DISTRIBUIÇÃO
--------------------------------------------------
Antes de 01/01/2020: 4,561,212 (58.11%)
De 01/01/2020 em diante: 3,288,670 (41.89%)


### Recorte temporal da análise

A base completa contém 7.849.882 registros, com contratações realizadas entre 02/01/2009 e 23/07/2026.

Para este projeto, optou-se por analisar as operações contratadas a partir de 01/01/2020.

O objetivo do recorte é concentrar a análise em um período mais recente das operações habitacionais, reduzindo a influência de períodos históricos potencialmente sujeitos a contextos econômicos, regras e condições de financiamento distintas.

O recorte preserva 3.288.670 registros, correspondentes a 41,89% da base original, mantendo volume expressivo de dados para análise e modelagem.

A influência do tempo sobre os dados ainda será considerada durante a análise exploratória e na estratégia de validação do modelo.

In [16]:
# Recorte temporal: operações contratadas a partir de 01/01/2020
DATA_INICIAL = pd.Timestamp("2020-01-01")

df = df.loc[
    df["data_assinatura_financiamento"] >= DATA_INICIAL
].copy()

df.reset_index(drop=True, inplace=True)

print("BASE APÓS O RECORTE TEMPORAL")
print("-" * 50)
print(f"Registros: {len(df):,}".replace(",", "."))
print(f"Variáveis: {df.shape[1]}")
print(
    "Primeira contratação:",
    df["data_assinatura_financiamento"].min().date()
)
print(
    "Última contratação:",
    df["data_assinatura_financiamento"].max().date()
)

memoria_gb = df.memory_usage(deep=True).sum() / (1024 ** 3)
print(f"Memória utilizada: {memoria_gb:.2f} GB")

C:\Users\c082909\AppData\Local\Temp\ipykernel_47352\1498732208.py:8: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.reset_index(drop=True, inplace=True)


BASE APÓS O RECORTE TEMPORAL
--------------------------------------------------
Registros: 3.288.670
Variáveis: 21
Primeira contratação: 2020-01-02
Última contratação: 2026-07-23
Memória utilizada: 1.86 GB


### 3.2 Estrutura e natureza das variáveis

Após o recorte temporal, serão analisados os tipos de dados e a cardinalidade das variáveis.

Essa análise é necessária porque o tipo armazenado no DataFrame não determina, isoladamente, a natureza estatística da variável. Uma variável numérica pode representar uma categoria ou identificador, enquanto uma variável textual pode demandar tratamento específico durante o pré-processamento.

Nesta etapa, nenhuma variável será excluída ou transformada.

In [17]:
resumo_variaveis = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "valores_unicos": df.nunique(dropna=True),
    "valores_nulos": df.isna().sum()
})

resumo_variaveis["percentual_nulos"] = (
    resumo_variaveis["valores_nulos"] / len(df) * 100
).round(2)

resumo_variaveis["percentual_unicos"] = (
    resumo_variaveis["valores_unicos"] / len(df) * 100
).round(4)

print("ESTRUTURA DAS VARIÁVEIS")
print("-" * 80)
print(f"Registros analisados: {len(df):,}".replace(",", "."))
print(f"Variáveis analisadas: {df.shape[1]}")

display(resumo_variaveis)

ESTRUTURA DAS VARIÁVEIS
--------------------------------------------------------------------------------
Registros analisados: 3.288.670
Variáveis analisadas: 21


,tipo_pandas,valores_unicos,valores_nulos,percentual_nulos,percentual_unicos
data_referencia,str,1,0,0.00,0.0000
cod_ibge,float64,4473,2,0.00,0.1360
txt_municipio,str,4266,2,0.00,0.1297
mcmv_fgts_txt_uf,str,27,2,0.00,0.0008
txt_regiao,str,5,2,0.00,0.0002
data_assinatura_financiamento,datetime64[us],1651,0,0.00,0.0502
qtd_uh_financiadas,int64,1,0,0.00,0.0000
vlr_financiamento,float64,1413968,0,0.00,42.9951
vlr_subsidio_desconto_fgts,float64,75125,0,0.00,2.2844
vlr_subsidio_desconto_ogu,float64,21664,0,0.00,0.6587
